# Libra Phase 4 — GPU extract + dev pilot

Runs on a free **16 GB GPU** (Kaggle or Colab).

**Stages:** `extract` (vector validation, fix L*) → `pilot` (8 dev personas) → `diagnose`.

When finished, download the `results/` zip and return it to the repo owner. Do **not** run test personas.

## 0. Settings

Set `REPO_URL` to the GitHub clone URL (or upload a zip and set `USE_UPLOADED_ZIP = True`).
Phi-3.5-mini is ungated; no HF token needed for the model. Keep secrets out of committed files.

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/harvathsteven/Libra.git"
REPO_DIR = Path("/kaggle/working/Libra")  # Colab: Path("/content/Libra")
BRANCH = "main"
STAGE = "phase4"  # or: steering_diag | extract | pilot | diagnose | figures
CONFIG = "configs/default.yaml"
USE_UPLOADED_ZIP = False
UPLOADED_ZIP = Path("/kaggle/input/libra-src/Libra.zip")  # adjust if using a dataset zip

# Optional: private clone via Kaggle secret GITHUB_TOKEN
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN") or (Path("/kaggle/input").exists() and None)
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    GITHUB_TOKEN = secrets.get_secret("GITHUB_TOKEN")
except Exception:
    pass

print("STAGE", STAGE)
print("REPO_DIR", REPO_DIR)

## 1. Check GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "Enable a GPU accelerator before running Phase 4"
print(torch.cuda.get_device_name(0))
print("mem_gb", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

## 2. Get the repository

In [ ]:
import shutil
import subprocess
import zipfile

if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    # Re-running in a live session: update to latest instead of silently using stale code.
    print("Repo already present; updating:", REPO_DIR)
    subprocess.check_call(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", BRANCH])
    subprocess.check_call(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{BRANCH}"])
    print("NOTE: if trait_mtmm was already imported, restart the kernel now.")
elif REPO_DIR.exists():
    print("Repo already present (no .git):", REPO_DIR)
elif USE_UPLOADED_ZIP:
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(UPLOADED_ZIP) as zf:
        zf.extractall(REPO_DIR.parent)
    # If zip contains a top-level Libra/, keep it; else rename
    if not REPO_DIR.exists():
        kids = list(REPO_DIR.parent.iterdir())
        print("Extracted:", kids)
else:
    url = REPO_URL
    if GITHUB_TOKEN and url.startswith("https://"):
        url = url.replace("https://", f"https://{GITHUB_TOKEN}@")
    subprocess.check_call(["git", "clone", "--branch", BRANCH, "--depth", "1", url, str(REPO_DIR)])

os.chdir(REPO_DIR)
print("cwd", Path.cwd())
print("has pyproject", (Path.cwd() / "pyproject.toml").exists())
print("has package", (Path.cwd() / "src" / "trait_mtmm").is_dir())
if (Path.cwd() / ".git").exists():
    print(subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())
else:
    print("no .git (zip install)")

## 3. Install dependencies

In [ ]:
import subprocess
import sys
from pathlib import Path

# Must be inside the repo root (has pyproject.toml + src/trait_mtmm)
assert Path("pyproject.toml").exists(), f"Not in repo root: {Path.cwd()}"
assert Path("src/trait_mtmm").is_dir(), f"Missing package dir in {Path.cwd()}"

# Kaggle kernels often miss editable installs; install deps then put src/ on sys.path.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "transformers>=4.44",
        "datasets",
        "accelerate",
        "numpy",
        "pandas",
        "scipy",
        "pyarrow",
        "matplotlib",
        "pyyaml",
        "pytest",
    ]
)
src = str((Path.cwd() / "src").resolve())
if src not in sys.path:
    sys.path.insert(0, src)

import trait_mtmm
from trait_mtmm.run import main

print("cwd", Path.cwd())
print("trait_mtmm", trait_mtmm.__version__, trait_mtmm.__file__)

## 4. Run the named stage

`steering_diag` = re-extract vectors + steering diagnostic (both label orders, split by keying, raw and ‖h‖-scaled coefficients). Not a gate.

`phase4` = extract (gates enforced; steering gate v2 = BC criterion, ‖h‖-scaled coefficients) + pilot (8 dev personas) + diagnose.


In [ ]:
import subprocess, sys
from trait_mtmm.run import main

if STAGE == "steering_diag":
    subprocess.check_call([sys.executable, "scripts/steering_diag.py", "--config", CONFIG])
else:
    main([STAGE, "--config", CONFIG])


## 5. Package outputs for download

In [ ]:
import shutil
from pathlib import Path

out_zip = Path("/kaggle/working/libra_phase4_results")
# Colab: out_zip = Path("/content/libra_phase4_results")
bundle = Path("/kaggle/working/bundle")
if bundle.exists():
    shutil.rmtree(bundle)
bundle.mkdir()
for name in ("results", "figures", "configs"):
    src = Path(name)
    if src.exists():
        shutil.copytree(src, bundle / name, dirs_exist_ok=True)
# Include updated default.yaml if L* was frozen
shutil.copy2("DECISIONS.md", bundle / "DECISIONS.md")
archive = shutil.make_archive(str(out_zip), "zip", bundle)
print("Download:", archive)
for p in sorted(Path("results").rglob("*")):
    if p.is_file():
        print(p, p.stat().st_size)

# Download link (FileLink needs a path relative to the notebook cwd)
import os
from IPython.display import FileLink, display
os.chdir("/kaggle/working")
display(FileLink(Path(archive).name))
os.chdir(REPO_DIR)  # keep later re-runs of cell 4 working
diag = Path("/kaggle/working/Libra/results/extract/steering_diag_table.csv")
if diag.exists():
    print(diag.read_text())


## Owner checklist after download

1. Unzip into the local repo (merge `results/`, `figures/`, and updated `configs/default.yaml`).
2. Confirm `results/extract/extraction_summary.json` has `all_gates_passed: true`.
3. Confirm `results/pilot/phase4_diagnostics.json` has `phase4_acceptance_passed: true`.
4. Paste key numbers into `DECISIONS.md` (L*, d′, option masses, any prompt-format change).
5. Only then proceed to Phase 5 (preregistration).